# Control Trainer

> Fill in a module description here

In [ ]:
#| default_exp trainers.control

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
import torch
import os
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.utils as vutils
import wandb
import hydra
from pathlib import Path
from fastcore.utils import patch
from loguru import logger
from omegaconf import DictConfig
from einops import rearrange
import torch.nn.functional as F
from c3jepa_wm.utils.checkpointer import RetrospectiveCheckpointer
from c3jepa_wm.utils import channel, PhaseTransitionChecker


In [ ]:
#| export
class TrainerScheduler:
    def __init__(self, wm_optimizer, power_optimizer, critic_optimizer):
        self.wm_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            wm_optimizer, mode='min', factor=0.5, patience=5
        )
        self.power_scheduler = self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                power_optimizer, mode='min', factor=0.5, patience=5
        )
        self.critic_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                critic_optimizer, mode='min', factor=0.5, patience=5
        )

    def step(self, val_loss, phase):

        if phase== 1 and 'val_jepa_loss' in val_loss:
            jepa_val_loss = val_loss['val_jepa_loss']
            self.wm_scheduler.step(jepa_val_loss)

        if phase == 2 and 'val_reward' in val_loss and 'val_critic_error' in val_loss:
            power_val_loss = val_loss['val_reward']
            self.power_scheduler.step(power_val_loss)

            critic_val_loss = val_loss['val_critic_error']
            self.critic_scheduler.step(critic_val_loss)

        if phase == 3 and 'val_jepa_loss' in val_loss and 'val_reward' in val_loss and 'val_critic_error' in val_loss:
            jepa_val_loss = val_loss['val_jepa_loss']
            self.wm_scheduler.step(jepa_val_loss)

            power_val_loss = val_loss['val_reward']
            self.power_scheduler.step(power_val_loss)

            critic_val_loss = val_loss['val_critic_error']
            self.critic_scheduler.step(critic_val_loss)
            
        

In [ ]:
#| export
class BaseTrainer:
    def __init__(self, 
                 data_module, 
                 device, 
                 slurm_jobid= None, 
                 wm_lr=1e-4, 
                 power_lr=3e-4,
                 epochs=100, 
                 project_name="c3jepa_wm",
                 ckp_dir= "checkpoints",
                 save_dir= "outputs"):
        
        self.data_module = data_module
        
        self.device = device
        
        self.data_module.setup()
        self.train_loader = self.data_module.train_dataloader()
        self.val_loader = self.data_module.val_dataloader()

        self.slurm_jobid = slurm_jobid if slurm_jobid else "default_job"
        self.wm_lr = wm_lr
        self.power_lr = power_lr
        self.epochs = epochs
        self.project_name = project_name
        self.ckp_dir = ckp_dir
        self.save_dir = save_dir

        
    def init_optimizer(self, model, lr, weight_decay=0.01):
        return torch.optim.AdamW(
                list(model.parameters()),
                lr=lr,
                weight_decay=weight_decay
            )
    
    def train_epoch(self, epoch):
        raise NotImplementedError("train_epoch method must be implemented by subclasses.")   

    def validate_epoch(self, epoch):
        raise NotImplementedError("validate method must be implemented by subclasses.")
    

    # def find_max_batch_size(self, phase):
    #     """Binary search for maximum batch size that fits in VRAM"""
    #     self.set_phase(phase)
    #     low, high = 1, 128
    #     best = 1

    #     while low <= high:
    #         mid = (low + high) // 2
    #         try:
    #             # Create a dummy batch of size mid
    #             dummy_batch = self._make_dummy_batch(mid)
    #             msg_indices = self.get_msg_indices(dummy_batch["sender_pov_seq"])
    #             msg_indices = rearrange(
    #                 msg_indices, "(b t) d -> b t d",
    #                 b=mid, t=self.history_size
    #             )

    #             if phase == 1:
    #                 self.train_epoch_phase_1(0, dummy_batch, msg_indices)
    #             elif phase == 2:
    #                 self.train_epoch_phase_2(0, dummy_batch, msg_indices)
    #             else:
    #                 self.train_epoch_phase_3(0, dummy_batch, msg_indices)

    #             best = mid
    #             low = mid + 1
    #             torch.cuda.empty_cache()

    #         except torch.cuda.OutOfMemoryError:
    #             high = mid - 1
    #             torch.cuda.empty_cache()

    #     print(f"Phase {phase}: max batch size = {best}")
    #     # Use 80% of max to leave headroom
    #     return int(best * 0.8)


    def find_max_batch_size(self, phase):
        if phase == 1:
            return 128
        elif phase == 2 or phase == 3:
            return 64

    def _make_dummy_batch(self, batch_size):
        T = self.history_size
        n_neighbors = 1
        return {
            "sender_pov_seq":  torch.randn(batch_size, T, 3, 224, 224).to(self.device),
            "receiver_pov_seq": torch.randn(batch_size, T + self.num_preds, 3, 224, 224).to(self.device),
            "receiver_act_seq": torch.zeros(batch_size, T, dtype=torch.long).to(self.device),
            "sender_csi":      torch.randn(batch_size, T, n_neighbors, 1, dtype=torch.complex64).to(self.device),
        }
        

    def _rebuild_dataloaders(self, phase):
            batch_size = self.phase_batch_sizes[phase]
            self.train_loader = DataLoader(
                self.data_module.train_dataset,
                batch_size=batch_size,
                shuffle=True,
                num_workers=0,
                pin_memory=True,
            )
            self.val_loader = DataLoader(
                self.data_module.val_dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=0,
                pin_memory=True,
            )
            print(f"Phase {phase}: batch size set to {batch_size}")

    def set_phase(self, phase):
        self.phase = phase
        if phase == 2:
            for param in self.model.parameters():
                param.requires_grad = False
        elif phase == 3:
            for param in self.model.parameters():
                param.requires_grad = True
            self.wm_optimizer = torch.optim.AdamW(
                self.model.parameters(), lr=1e-5
            )
        # Rebuild dataloaders with new batch size
        self._rebuild_dataloaders(phase)


In [ ]:
#| export
class WMTrainer(BaseTrainer):
    def __init__(self, data_module, model, device, slurm_jobid, wm_lr, power_lr, history_size, num_preds, lambda_sigreg, lambda_pow, lambda_value, lambda_quality, lambda_send, **kwargs):
        super().__init__(
            data_module= data_module,
            device= device, 
            wm_lr= wm_lr,
            slurm_jobid= slurm_jobid,
            power_lr= power_lr,
            **kwargs)
        
        self.history_size = history_size
        self.num_preds = num_preds
        self.phase = 1
        self.phase_batch_sizes = {}
        for phase in [1, 2, 3]:
            bs = self.find_max_batch_size(phase)
            self.phase_batch_sizes[phase] = bs
        self.min_phase_epochs = {1: 1, 2: 5}

        self.lambda_sigreg = lambda_sigreg
        self.lambda_pow = lambda_pow
        self.lambda_value = lambda_value
        self.lambda_quality = lambda_quality
        self.lambda_send = lambda_send

        self.vqvae = model["vqvae"].to(device)
        self.power_net = model["power_net"].to(device)
        self.model = model["jepa"].to(device)
        self.critic = model['critic'].to(device)

        self.model.predictor.msg_token_embedding.weight.data[:self.vqvae.vq_layer.K].copy_(
            self.vqvae.vq_layer.embedding.detach()
        )
        self.power_net.msg_embedding.weight.data.copy_(
            self.vqvae.vq_layer.embedding.detach()
        )
        self.critic.msg_embedding.weight.data.copy_(
            self.vqvae.vq_layer.embedding.detach()
        )

        self.wm_optimizer = self.init_optimizer(self.model, lr=self.wm_lr, weight_decay=1e-3)
        self.power_optimizer = self.init_optimizer(self.power_net, lr=self.power_lr, weight_decay=1e-4)
        self.critic_optimizer = self.init_optimizer(self.critic, lr=self.power_lr, weight_decay=1e-4)

        self.scheduler = TrainerScheduler(self.wm_optimizer, self.power_optimizer, self.critic_optimizer)
        
        self.save_dir = Path(hydra.utils.to_absolute_path(self.save_dir))
        self.save_dir.mkdir(exist_ok=True, parents=True)

        self.ck_pointer = RetrospectiveCheckpointer(project_name= self.project_name,
                                                    ckp_dir= self.ckp_dir,
                                                    slurm_jobid= self.slurm_jobid,
                                                    agent_id= "vqvae_trainer",
                                                    rank= 0, 
                                                    n_best=3)
        
        self.transition_checker = PhaseTransitionChecker(
            plateau_patience=2,
            plateau_threshold=1e-3,
            reward_patience=5,
            schedule_rate_min=0.1,
            schedule_rate_max=0.9,
        )

        # Create output directories for visual inspection
        os.makedirs(os.path.join(self.save_dir, "Reconstructions"), exist_ok=True)



In [ ]:
#| export
@patch
def fit(self: WMTrainer, cfg: DictConfig):
    self.set_phase(1)
    for epoch in range(1, cfg.pipeline.max_epochs + 1):
        train_loss = self.train_epoch(epoch)
        metrics = self.validate_epoch(epoch)
        self.scheduler.step(metrics, self.phase)
        self.checkpoint(epoch, metrics)
        
        self.transition_checker.update(metrics)

        if self.phase == 1:
            if (epoch >= self.min_phase_epochs[1] and self.transition_checker.should_transition_to_phase2()):
                self.set_phase(2)
                self.transition_checker.reset()  # fresh history for phase 2

        elif self.phase == 2:
            if (epoch >= self.min_phase_epochs[2] and self.transition_checker.should_transition_to_phase3()):
                self.set_phase(3)
                self.transition_checker.reset()
        

In [ ]:
#| export
@patch
@torch.no_grad()
def compute_rewards(self: WMTrainer, ctx_emb, ctx_act, tgt_emb, received_msg):
    """Takes pre-computed embeddings instead of re-encoding"""
    T = ctx_emb.shape[1]
    ctx_msg = received_msg[:, :T]

    pred_emb_no_msg = self.model.predict(ctx_emb, ctx_act, None)
    pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg)

    baseline_loss = (pred_emb_no_msg - tgt_emb).pow(2).mean()
    pred_loss = (pred_emb - tgt_emb).pow(2).mean()
    reward = baseline_loss - pred_loss
    return reward, pred_loss, baseline_loss

In [ ]:
#| export
@patch
@torch.no_grad()
def get_msg_indices(self: WMTrainer, sender_pov_seq):
    """Helper function to get message indices from the pretrained VQ-VAE for a given sender POV sequence.
    Args:
        sender_pov_seq: Tensor of shape (B, T, C, H, W) representing the sender's point-of-view image sequence.
    Returns:
        msg_indices: Tensor of shape (B*T, 49) 
    """
    B, T, C, H, W = sender_pov_seq.shape
    sender_pov_flat = rearrange(sender_pov_seq.to(self.device), "b t c h w -> (b t) c h w")  # (B*T, C, H, W)
    msg_indices_flat = self.vqvae.get_message_indices(sender_pov_flat)  # (B*T, 7, 7)
    msg_indices = rearrange(msg_indices_flat, "b h w -> b (h w)", b=B*T)   # (B*T, 49)
    return msg_indices



In [ ]:
#| export
@patch
def train_epoch(self: WMTrainer, epoch):
    self.model.train()
    self.power_net.train()
    self.critic.train()

    total_loss_jepa = 0.0
    total_loss_power = 0.0

    for batch_idx, batch in enumerate(self.train_loader):
        batch = {k: v.to(self.device) for k, v in batch.items()}

        # 1. Get message indices from pretrained VQ-VAE
        msg_indices = self.get_msg_indices(
            batch["sender_pov"]
        )  # (B*T, 49)

        if self.phase == 1:
            jepa_loss, tx_loss = self.train_epoch_phase_1(epoch, batch, msg_indices)

        elif self.phase == 2:
            jepa_loss, tx_loss = self.train_epoch_phase_2(epoch, batch, msg_indices)
        else:
            jepa_loss, tx_loss = self.train_epoch_phase_3(epoch, batch, msg_indices)
            
        total_loss_jepa += jepa_loss
        total_loss_power += tx_loss

        logger.info(f"Completed batch {batch_idx+1}/{len(self.train_loader)} for epoch {epoch} with JEPA loss {jepa_loss:.4f} and Power loss {tx_loss:.4f}")

    avg_loss_jepa = total_loss_jepa / len(self.train_loader)
    avg_loss_power = total_loss_power / len(self.train_loader)

    logger.info(f"Epoch [{epoch}/{self.epochs}] - Train JEPA Loss: {avg_loss_jepa:.4f}, Power Loss: {avg_loss_power:.4f}")

    # wandb.log({"train_avg_jepa_loss": avg_loss_jepa, "train_avg_power_loss": avg_loss_power}, step=epoch)
    return avg_loss_jepa, avg_loss_power


In [ ]:
#| export
@patch
def train_epoch_phase_1(self: WMTrainer, epoch, batch, msg_indices):
    B = batch["sender_pov"].shape[0]
    T = self.history_size
    self.wm_optimizer.zero_grad()

    msg_indices = rearrange(msg_indices, "(b t) n -> b t n", b=B, t=T, n= 49)  # flatten batch and time for embedding

    output = self.model.encode(batch)
    emb = output["emb"]
    act_emb = output["act_emb"].reshape(B, T, -1)
    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    ctx_msg = msg_indices[:, :T]          # perfect message
    tgt_emb = emb[:, self.num_preds:]

    pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg)
    output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)

    loss_dict = {k: v for k, v in output.items() if "loss" in k}
    wandb.log({f"train_{k}": v.item() for k, v in loss_dict.items()})

    output['jepa_loss'].backward()
    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    self.wm_optimizer.step()
    return output['jepa_loss'].item(), 0.0


In [ ]:
#| export
@patch
def train_epoch_phase_2(self: WMTrainer, epoch, batch, msg_indices):
    B = batch["sender_pov"].shape[0]
    T = self.history_size
    self.power_optimizer.zero_grad()
    self.critic_optimizer.zero_grad()

    csi_flat = rearrange(
        batch["sender_csi"].to(self.device), "b t n d -> (b t) n d"
    )

    schedule, power = self.power_net(msg_indices, csi_flat)

    received_msg = channel(schedule, power, msg_indices, csi_flat, device=self.device)
    received_msg = rearrange(received_msg, "(b t) msg_dim -> b t msg_dim", b=B, t=T)

    # Encode once, reuse in compute_rewards
    with torch.no_grad():
        output = self.model.encode(batch)
        emb = output["emb"]
        act_emb = output["act_emb"].reshape(B, T, -1)
        ctx_emb = emb[:, :T]
        ctx_act = act_emb[:, :T]
        tgt_emb = emb[:, self.num_preds:]

    reward, pred_loss, baseline_loss = self.compute_rewards(
        ctx_emb, ctx_act, tgt_emb, received_msg
    )

    tx_loss, critic_loss, actor_loss = self.power_net.loss_fn(
        self.critic, reward, msg_indices, csi_flat, schedule, power
    )

    wandb.log({
        "train_tx_loss": tx_loss.item(),
        "train_critic_loss": critic_loss.item(),
        "train_actor_loss": actor_loss.item(),
        "train_pred_loss": pred_loss.item(),
        "train_baseline_loss": baseline_loss.item(),
        "train_reward": reward.item(),
    })

    tx_loss.backward()
    torch.nn.utils.clip_grad_norm_(self.power_net.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=1.0)
    self.power_optimizer.step()
    self.critic_optimizer.step()
    return 0.0, tx_loss.item()

In [ ]:
#| export
@patch
def train_epoch_phase_3(self: WMTrainer, epoch, batch, msg_indices):
    B = batch["sender_pov"].shape[0]
    T = self.history_size
    
    self.wm_optimizer.zero_grad()
    self.power_optimizer.zero_grad()
    self.critic_optimizer.zero_grad()

    csi_flat = rearrange(
        batch["sender_csi"].to(self.device), "b t n d -> (b t) n d"
    )

    schedule, power = self.power_net(msg_indices, csi_flat)

    received_msg = channel(schedule, power, msg_indices, csi_flat, device=self.device)
    received_msg = rearrange(received_msg, "(b t) msg_dim -> b t msg_dim", b=B, t=T)

    # JEPA forward with noisy received message
    output = self.model.encode(batch)
    emb = output["emb"]
    act_emb = output["act_emb"].reshape(B, T, -1)
    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    ctx_msg = received_msg[:, :T]         # ← noisy, not perfect
    tgt_emb = emb[:, self.num_preds:]

    pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg)
    output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)

    # Compute reward using already-computed embeddings
    reward, pred_loss, baseline_loss = self.compute_rewards(
        ctx_emb.detach(), ctx_act.detach(), tgt_emb.detach(), received_msg
    )

    tx_loss, critic_loss, actor_loss = self.power_net.loss_fn(
        self.critic, reward, msg_indices, csi_flat, schedule, power
    )

    wandb.log({
        "train_jepa_loss": output['jepa_loss'].item(),
        "train_tx_loss": tx_loss.item(),
        "train_critic_loss": critic_loss.item(),
        "train_actor_loss": actor_loss.item(),
        "train_reward": reward.item(),
    })

    # Correct backward order
    output['jepa_loss'].backward(retain_graph=True)
    tx_loss.backward()

    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(self.power_net.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=1.0)

    self.wm_optimizer.step()
    self.power_optimizer.step()
    self.critic_optimizer.step()
    return output['jepa_loss'].item(), tx_loss.item()

In [ ]:
#| export
@patch
@torch.no_grad()
def validate_epoch(self: WMTrainer, epoch):
    self.model.eval()
    self.power_net.eval()
    self.critic.eval()

    all_metrics = {}
    for batch in self.val_loader:
        batch = {k: v.to(self.device) if torch.is_tensor(v) else v 
                for k, v in batch.items()}
        msg_indices = self.get_msg_indices(batch["sender_pov"])
        
        if self.phase == 1:
            metrics = self.validate_phase_1(batch, msg_indices)
        elif self.phase == 2:
            metrics = self.validate_phase_2(batch, msg_indices)
        else:
            metrics = self.validate_phase_3(batch, msg_indices)

        for k, v in metrics.items():
            all_metrics.setdefault(k, []).append(v)

    # Average over all batches
    avg_metrics = {k: sum(v) / len(v) for k, v in all_metrics.items()}
    wandb.log({**avg_metrics, "epoch": epoch})

    self.model.train()
    self.power_net.train()
    self.critic.train()

    return avg_metrics


In [ ]:
#| export
@patch
@torch.no_grad()
def validate_phase_1(self: WMTrainer, batch, msg_indices):
    B = batch["sender_pov"].shape[0]
    T = self.history_size

    msg_indices = rearrange(msg_indices, "(b t) n -> b t n", b=B, t=T, n= 49)  # flatten batch and time for embedding

    output = self.model.encode(batch)
    emb = output["emb"]
    act_emb = output["act_emb"].reshape(B, T, -1)
    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    ctx_msg = msg_indices[:, :T]          # perfect message
    tgt_emb = emb[:, self.num_preds:]

    pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg)
    output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)

    return {"val_jepa_loss": output['jepa_loss'].item()}

In [ ]:
#| export
@patch
@torch.no_grad()
def validate_phase_2(self: WMTrainer, batch, msg_indices):
    B = batch["sender_pov"].shape[0]
    T = self.history_size

    csi_flat = rearrange(
        batch["sender_csi"].to(self.device), "b t n d -> (b t) n d"
    )

    schedule, power = self.power_net(msg_indices, csi_flat)

    received_msg = channel(schedule, power, msg_indices, csi_flat, device=self.device)
    received_msg = rearrange(received_msg, "(b t) msg_dim -> b t msg_dim", b=B, t=T)

    output = self.model.encode(batch)
    emb = output["emb"]
    act_emb = output["act_emb"].reshape(B, T, -1)
    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    tgt_emb = emb[:, self.num_preds:]

    reward, pred_loss, baseline_loss = self.compute_rewards(
        ctx_emb, ctx_act, tgt_emb, received_msg
    )

    # Also evaluate critic accuracy
    values = self.critic(msg_indices, csi_flat)
    critic_error = F.mse_loss(values, reward.expand_as(values))

    return {
        "val_reward":        reward.item(),
        "val_pred_loss":     pred_loss.item(),
        "val_baseline_loss": baseline_loss.item(),
        "val_critic_error":  critic_error.item(),
        "val_schedule_rate": schedule.mean().item(),  # how often it decides to send
        "val_power_mean":    power.mean().item(),
    }

In [ ]:
#| export
@patch
@torch.no_grad()
def validate_phase_3(self: WMTrainer, batch, msg_indices):
    B = batch["sender_pov"].shape[0]
    T = self.history_size

    csi_flat = rearrange(
        batch["sender_csi"].to(self.device), "b t n d -> (b t) n d"
    )

    schedule, power = self.power_net(msg_indices, csi_flat)

    received_msg = channel(schedule, power, msg_indices, csi_flat, device=self.device)
    received_msg = rearrange(received_msg, "(b t) msg_dim -> b t msg_dim", b=B, t=T)

    output = self.model.encode(batch)
    emb = output["emb"]
    act_emb = output["act_emb"].reshape(B, T, -1)
    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    ctx_msg = received_msg[:, :T]         # noisy received message
    tgt_emb = emb[:, self.num_preds:]

    pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg)
    output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)

    reward, pred_loss, baseline_loss = self.compute_rewards(
        ctx_emb, ctx_act, tgt_emb, received_msg
    )

    # Also evaluate critic accuracy
    values = self.critic(msg_indices, csi_flat)
    critic_error = F.mse_loss(values, reward.expand_as(values))

    # Also compare against no-comm and perfect-comm baselines
    msg_indices = rearrange(msg_indices, "(b t) n -> b t n", b=B, t=T, n= 49)  # flatten batch and time for embedding
    pred_emb_perfect = self.model.predict(ctx_emb, ctx_act, msg_indices[:, :T])
    perfect_loss = (pred_emb_perfect - tgt_emb).pow(2).mean()

    return {
        "val_jepa_loss":     output['jepa_loss'].item(),
        "val_pred_loss":     pred_loss.item(),
        "val_baseline_loss": baseline_loss.item(),   # no comm
        "val_perfect_loss":  perfect_loss.item(),    # upper bound
        "val_reward":        reward.item(),
        "val_schedule_rate": schedule.mean().item(),
        "val_power_mean":    power.mean().item(),
        "val_critic_error":  critic_error.item(),
    }


In [ ]:
#| export
@patch
def checkpoint(self: WMTrainer, epoch, val_loss):

    if self.phase == 1:
        checkpoint_state = {
        "epoch": epoch,
        "wm_optimizer_state_dict": self.wm_optimizer.state_dict(),
        "wm_model_state_dict": self.model.state_dict(),
        "val_jepa_loss": val_loss['val_jepa_loss']
        }

    elif self.phase == 2:
         checkpoint_state = {
        "epoch": epoch,
        "power_optimizer_state_dict": self.power_optimizer.state_dict(),
        "power_model_state_dict": self.power_net.state_dict(),
        "critic_model_state_dict": self.critic.state_dict(),
        "critic_optimizer_state_dict": self.critic_optimizer.state_dict(),
        "val_reward": val_loss['val_reward'],
        "critic_error": val_loss['val_critic_error']
        }
         
    else:
        checkpoint_state = {
            "epoch": epoch,
            "wm_model_state_dict": self.model.state_dict(),
            "wm_optimizer_state_dict": self.wm_optimizer.state_dict(),
            "power_model_state_dict": self.power_net.state_dict(),
            "power_optimizer_state_dict": self.power_optimizer.state_dict(),
            "critic_model_state_dict": self.critic.state_dict(),
            "critic_optimizer_state_dict": self.critic_optimizer.state_dict(),
            "val_jepa_loss": val_loss['val_jepa_loss'],
            "val_reward": val_loss['val_reward']
        }

    if self.phase == 1:
        acc = -val_loss["val_jepa_loss"]
    else:
        acc = val_loss["val_reward"]


    self.ck_pointer.save_checkpoint(state= checkpoint_state, current_acc= acc, step= epoch)
    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()